### Structured output


In [11]:

import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model =init_chat_model("groq:llama-3.1-8b-instant")
model


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7bc879ed9fd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7bc879e2ab70>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [12]:
from pydantic import BaseModel,Field
class Movies(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

In [13]:
model_with_structure=model.with_structured_output(Movies)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7bc879ed9fd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7bc879e2ab70>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movies', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out of

In [15]:
model.invoke("provide the  details about the movie inception ")

AIMessage(content='**Inception (2010)**\n\n**Directed by:** Christopher Nolan\n**Produced by:** Christopher Nolan, Charles Roven, Emma Thomas\n**Written by:** Christopher Nolan\n**Cinematography:** Wally Pfister\n**Edited by:** Lee Smith\n**Music by:** Hans Zimmer\n\n**Plot Summary:**\n\nInception is a mind-bending science fiction action film that explores the concept of shared dreaming. The movie follows the story of Cobb (Leonardo DiCaprio), a skilled thief who specializes in entering people\'s dreams and stealing their secrets. Cobb is hired by a wealthy businessman named Saito (Ken Watanabe) to perform a task known as "inception" - planting an idea in someone\'s mind instead of stealing one.\n\nSaito wants Cobb to convince Robert Fischer (Cillian Murphy), the son of a dying business magnate, to dissolve his father\'s company. In exchange, Saito will wipe Cobb\'s murder conviction off the books, allowing him to return to the United States and reunite with his children.\n\nCobb assem

In [17]:
model_with_structure.invoke("provide the  details about the movie inception")

Movies(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [18]:
model_with_structure=model.with_structured_output(Movies, include_raw=True)
model_with_structure
model_with_structure.invoke("provide the  details about the movie inception ")


{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'vecnh4efx', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.5,"title":"Inception","year":2010}', 'name': 'Movies'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 281, 'total_tokens': 314, 'completion_time': 0.043788536, 'completion_tokens_details': None, 'prompt_time': 0.631015244, 'prompt_tokens_details': None, 'queue_time': 0.282933809, 'total_time': 0.67480378}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec5b2-afa8-7fc1-b8ba-a863a54cdbd9-0', tool_calls=[{'name': 'Movies', 'args': {'director': 'Christopher Nolan', 'rating': 8.5, 'title': 'Inception', 'year': 2010}, 'id': 'vecnh4efx', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 281, 'output_tokens': 33, '

### Nested structure 

In [22]:
from pydantic import BaseModel,Field
from typing import List

class Actor(BaseModel):
 name:str
 role:str | None = None

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:List[Actor]
    genres:List[str]
    budget:float | None = Field(None,description="Budget in million USD")

In [23]:
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("provide the  details about the movie inception ")
response


MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role=None), Actor(name='Joseph Gordon-Levitt', role=None), Actor(name='Ellen Page', role=None), Actor(name='Tom Hardy', role=None), Actor(name='Ken Watanabe', role=None), Actor(name='Dileep Rao', role=None), Actor(name='Cillian Murphy', role=None), Actor(name='Tom Berenger', role=None), Actor(name='Pete Postlethwaite', role=None)], genres=['Action', 'Sci-Fi', 'Thriller', 'Mystery'], budget=160.0)

In [24]:
from typing_extensions import TypedDict,Annotated


In [25]:
class Moviedict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ...,"the title of the movie"]
    year: Annotated[int, ...,"The year the movie was released"]
    director: Annotated[str, ...,"The diretor of the movie"]
    rating: Annotated[float, ...,"The movies rating out of 10"]

In [32]:
model_withtypedict=model.with_structured_output(Moviedict)
response=model_withtypedict.invoke("please provide the detsils of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [33]:
class Actor(TypedDict):
 name:str
 role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:List[Actor]
    genres:List[str]
    budget:float | None = Field(None,description="Budget in million USD")

model_with_structure = model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("please provide the detsils of the movie avengers")
response   

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'Avengers',
 'year': 2012}

In [34]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 8192,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True}

### DataClasses


In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""

    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)

print(result)

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='7ae2550a-851d-4989-b76b-3989ff68c796'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 't2s3rn7hy', 'function': {'arguments': '{"email":"john@example.com","name":"John Doe","phone":"(555) 123-4567"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 288, 'total_tokens': 319, 'completion_time': 0.031939541, 'completion_tokens_details': None, 'prompt_time': 0.0183389, 'prompt_tokens_details': None, 'queue_time': 0.159359666, 'total_time': 0.050278441}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec5bb-8ef1-7ef3-8ccb-678eead0b745-0', tool_calls=[{'name': 'ContactInfo', 'args': {'email': 'john@exa

In [37]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [39]:
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""

    name: str  # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person


agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)

print(result["structured_response"])
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


In [40]:
from dataclasses import dataclass
from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information for a person."""

    name: str  # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person


agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)

print(result["structured_response"])
# ContactInfo(
#     name='John Doe',
#     email='john@example.com',
#     phone='(555) 123-4567'
# )

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')
